# 02 — LLaMA Architecture Lab

Prototype and explore the **LLaMA** architecture using the building blocks in
`src/core/`.  This is where you iterate before promoting to `src/`.

In [1]:
import sys
sys.path.insert(0, '../..')

import torch
from src.models.llama.config import LlamaConfig
from src.models.llama.model import LlamaSLM
from src.core.generation import generate
from src.utils.training import count_params

In [2]:
# ── Build a tiny model for fast iteration ────────────────────────────────
tiny_cfg = LlamaConfig(
    vocab_size=50257, block_size=32,
    n_layer=2, n_head=2, n_kv_head=1, n_embd=64,
    intermediate_size=128, dropout=0.0,
)
model = LlamaSLM(tiny_cfg)
print(f'Parameters: {count_params(model):,}')

Parameters: 3,290,496


In [3]:
# ── Verify forward pass shapes ────────────────────────────────────────────
B, T = 2, tiny_cfg.block_size
idx = torch.randint(0, tiny_cfg.vocab_size, (B, T))
targets = torch.randint(0, tiny_cfg.vocab_size, (B, T))

logits, loss = model(idx, targets)
print(f'logits: {logits.shape}  |  loss: {loss.item():.4f}')

logits: torch.Size([2, 32, 50257])  |  loss: 10.8253


In [4]:
# ── Test generation ───────────────────────────────────────────────────────
prompt = torch.randint(0, tiny_cfg.vocab_size, (1, 4))
output = generate(model, prompt, max_new_tokens=10, temperature=1.0, top_k=50)
print(f'Input tokens:  {prompt[0].tolist()}')
print(f'Output tokens: {output[0].tolist()}')

Input tokens:  [5386, 22625, 6245, 15397]
Output tokens: [5386, 22625, 6245, 15397, 1557, 41404, 15862, 14640, 2454, 28885, 36949, 36063, 21859, 6441]


In [5]:
# ── Parameter count comparison across model sizes ────────────────────────
configs = {
    'llama_tiny':   LlamaConfig(vocab_size=50257, block_size=16,  n_layer=2,  n_head=2,  n_kv_head=1, n_embd=64,  intermediate_size=128),
    'llama_small':  LlamaConfig(vocab_size=50257, block_size=128, n_layer=6,  n_head=6,  n_kv_head=2, n_embd=384, intermediate_size=1024),
    'llama_medium': LlamaConfig(vocab_size=50257, block_size=256, n_layer=12, n_head=12, n_kv_head=4, n_embd=768, intermediate_size=2048),
}
for name, cfg in configs.items():
    n = count_params(LlamaSLM(cfg))
    print(f'{name:14s}: {n:>12,} params  ({n/1e6:.1f}M)')

llama_tiny    :    3,290,496 params  (3.3M)
llama_small   :   28,740,864 params  (28.7M)
llama_medium  :  114,114,048 params  (114.1M)
